In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">h3. 연관분석</font>**
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이의 자주 발생하는 속성을 찾고, 그 속성들 사이의 연관성이 어느 정도 있는지를 분석
- 활용분야 : 상품진열, 사기보험적발, 신상품 카테고리 구성,...
```
조건(left-hand side, item_base) : 오렌지주스(x) => 결과(right-hand side, item_add) : 와인(y)

연관분석 지표
1. 지지도(support) : 전체 데이터 중, 조건과 결과 항목들이 포함된 거래 비율(함께 얼마나 자주 나타나는지)
    (x, y)의 항목수 / 전체 데이터수 = 0.2
2. 신뢰도(confidence) : 조건(x)이 발생했을 때, 결과가 동시에 일어날 확률(조건이 오면 얼마나 자주 결과가 오는지)
    (x=>y)의 항목수 / x가 나오는 항목수 = 0.5
3. 향상도(lift) : 우연히 발생할 규칙은 아니였는지 확인
    1미만 : 독립적으로 나오는 것보다 함께 나타날 가능성이 낮다
    1    : 서로 독립적. 아무 연관성 없다
    1초과 : 양의 상관관계(같이 잘 나온다)
    (x=>y)의 지지도 / (x의 지지도*y의 지지도) = 0.2 / (0.4*0.6) = 0.2/0.24 = 0.83333
```

# 2. 연관분석 구현

In [2]:
import csv
with open('data/cf_basket.csv', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [9]:
from apyori import apriori
rules = apriori(transaction,
               min_support=0.15,
               min_confidence=0.1,
               min_lift=1.001)
rules = list(rules)
len(rules)

6

In [12]:
rule = rules[5]
rule

RelationRecord(items=frozenset({'소주', '콜라', '와인'}), support=0.2, ordered_statistics=[OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주', '와인'}), confidence=0.25, lift=1.25), OrderedStatistic(items_base=frozenset({'소주', '와인'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25)])

In [23]:
support = rule[1]
ordered_st = rule[2]
for item in ordered_st:
    # print(item)
    lhs = item[0]
    lhs = ','.join([x for x in lhs])
    rhs = item[1]
    rhs = ','.join([x for x in rhs])
    confidence = item[2]
    lift = item[3]
    print(f"{lhs}=>{rhs} \t {support} \t {confidence} \t {lift}")

콜라=>소주,와인 	 0.2 	 0.25 	 1.25
소주,와인=>콜라 	 0.2 	 1.0 	 1.25
